In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow_hub as hub
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
# Load the dataset
df = pd.read_excel('/content/hec-sim.xlsx')

In [3]:
df.head(2)

,Id,Full,Group,class,invCnt,greens,reds,controv,winning
0,39,Improve Elective Selection and Choice Process ...,A,highPriority,10,10,3,0.488770,NaN
1,40,Assign alumni mentors : All HEC Paris MBA stud...,A,neutral,11,9,3,0.698242,NaN


In [4]:
# Load the Universal Sentence Encoder
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

In [5]:
# Encode all ideas
embeddings = embed(df['Full'].tolist())

In [17]:
#embeddings_np = np.array(embeddings)
#df['embedding'] = embeddings_np.tolist()
#df.to_csv('use_embeddings.csv', index=False)

In [6]:
# Compute similarity scores
similarity_matrix = cosine_similarity(embeddings)

In [7]:
def find_most_similar_same_group(idx, similarity_matrix, df):
    current_group = df.loc[idx, 'Group']
    same_group_indices = df[df['Group'] == current_group].index
    similarities = similarity_matrix[idx, same_group_indices]
    # Set similarity to itself as -1 to exclude it from consideration
    similarities[same_group_indices == idx] = -1
    most_similar_idx = same_group_indices[np.argmax(similarities)]
    return df.loc[most_similar_idx, 'Id'], similarities[np.argmax(similarities)]

# Function to find most similar idea in the other group
def find_most_similar_other_group(idx, similarity_matrix, df):
    current_group = df.loc[idx, 'Group']
    other_group = 'B' if current_group == 'A' else 'A'
    other_group_indices = df[df['Group'] == other_group].index
    similarities = similarity_matrix[idx, other_group_indices]
    most_similar_idx = other_group_indices[np.argmax(similarities)]
    return df.loc[most_similar_idx, 'Id'], similarities[np.argmax(similarities)]

In [8]:
# Find most similar ideas and add to dataframe
df['SimilarSameGroup'] = ''
df['CsSame'] = 0.0
df['SimilarOtherGroup'] = ''
df['CsOther'] = 0.0

for idx in df.index:
    similar_same, score_same = find_most_similar_same_group(idx, similarity_matrix, df)
    similar_other, score_other = find_most_similar_other_group(idx, similarity_matrix, df)

    df.at[idx, 'MostSimilarSameGroup'] = similar_same
    df.at[idx, 'SimilarityScoreSameGroup'] = score_same
    df.at[idx, 'MostSimilarOtherGroup'] = similar_other
    df.at[idx, 'SimilarityScoreOtherGroup'] = score_other

In [9]:
df.head()

,Id,Full,Group,class,invCnt,greens,reds,controv,winning,SimilarSameGroup,CsSame,SimilarOtherGroup,CsOther,MostSimilarSameGroup,SimilarityScoreSameGroup,MostSimilarOtherGroup,SimilarityScoreOtherGroup
0,39,Improve Elective Selection and Choice Process ...,A,highPriority,10,10,3,0.488770,NaN,,0.0,,0.0,165.0,0.498870,98.0,0.613414
1,40,Assign alumni mentors : All HEC Paris MBA stud...,A,neutral,11,9,3,0.698242,NaN,,0.0,,0.0,116.0,0.444855,193.0,0.445510
2,43,Improve access to Alumni network incl. Dynamic...,A,highPriority,15,14,2,0.031128,NaN,,0.0,,0.0,111.0,0.363480,105.0,0.643764
3,49,"Enhance professor diversity (gender, sectors) ...",A,controversial,21,14,10,2.922499,NaN,,0.0,,0.0,133.0,0.470688,114.0,0.601659
4,52,Access to past student reviews on elective mod...,A,highPriority,18,14,5,0.443573,1.0,,0.0,,0.0,39.0,0.456452,21.0,0.393103


In [10]:
print(df['class'].unique())

['highPriority' 'neutral' 'controversial' 'lowPriority']
